# RAG retriever v3

В данном ноутбуке:

1. Скачиваются актуальные RAG-артефакты из S3 полученные в ноутбуке `RAG_baseline-2.ipynb`.
2. Происходит подключение к Postgres БД и собирается `database_schema` corpus из комментариев таблиц и их колонок.
3. После сбора корпуса происходит обогащение общего корпуса базы знаний для RAG retriever.
4. Пересобираются эмбеддинги и проверяется retrieval в двух режимах:
   - global retrieval по всем корпусам;
   - schema-only retrieval для SQL-сценария.
5. Происходит попытка улучшения retriever на сгенерированном по `database_schema` corpus.
6. Сохраняются артефакты той же структуры в виде архива в S3-хранилище для сервиса.


In [ ]:
!pip install -q \
  boto3 botocore \
  psycopg2-binary \
  pandas numpy tqdm joblib pyarrow \
  sentence-transformers datasets scikit-learn \
  sqlglot pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 146.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.1 MB/s eta 0:00:00


## 1. Импорты и базовые настройки


In [ ]:
import os
import json
import shutil
import zipfile
import random
import gc
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib
from tqdm.auto import tqdm

import boto3
from botocore.client import Config

import psycopg2
from psycopg2.extras import RealDictCursor

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from sentence_transformers import SentenceTransformer, InputExample, losses

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEMORY_GB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    GPU_NAME = "Apple Silicon MPS"
    GPU_MEMORY_GB = None
else:
    DEVICE = "cpu"
    GPU_NAME = "CPU"
    GPU_MEMORY_GB = None

USE_CUDA = DEVICE == "cuda"
USE_GPU = DEVICE in {"cuda", "mps"}
USE_AMP = USE_CUDA

DEFAULT_EMBED_BATCH_SIZE = 128 if USE_CUDA else (64 if DEVICE == "mps" else 32)
TRAIN_BATCH_SIZE_DEFAULT = 32 if USE_CUDA else (16 if DEVICE == "mps" else 16)
DATALOADER_NUM_WORKERS = 2 if USE_CUDA else 0
PIN_MEMORY = USE_CUDA

WORK_DIR = Path("rag_retriever_v2_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS_DIR = WORK_DIR / "artifacts_current"
NEW_ARTIFACTS_DIR = WORK_DIR / "artifacts_rag_baseline_latest"

SCHEMA = "rag_kg"
TOP_K_VALUES = (1, 3, 5, 10)

print("WORK_DIR:", WORK_DIR.resolve())
print("torch:", torch.__version__)
print("DEVICE:", DEVICE)
print("GPU:", GPU_NAME)
if GPU_MEMORY_GB is not None:
    print("GPU memory GB:", GPU_MEMORY_GB)
print("DEFAULT_EMBED_BATCH_SIZE:", DEFAULT_EMBED_BATCH_SIZE)
print("TRAIN_BATCH_SIZE_DEFAULT:", TRAIN_BATCH_SIZE_DEFAULT)
print("USE_AMP:", USE_AMP)


WORK_DIR: /content/rag_retriever_v2_work
torch: 2.10.0+cu128
DEVICE: cuda
GPU: Tesla T4
GPU memory GB: 14.56
DEFAULT_EMBED_BATCH_SIZE: 128
TRAIN_BATCH_SIZE_DEFAULT: 32
USE_AMP: True


/tmp/ipykernel_18891/2316207746.py:26: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


## 2. Секреты и конфигурация


Ожидаемые переменные:

- `S3_ENDPOINT_URL`
- `S3_BUCKET`
- `S3_ARTIFACT_KEY`
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_DEFAULT_REGION`
- `POSTGRES_HOST`
- `POSTGRES_PORT`
- `POSTGRES_DB`
- `POSTGRES_USER`
- `POSTGRES_PASSWORD`
- `POSTGRES_SSLMODE`


In [ ]:
try:
    from google.colab import userdata

    for key in [
        "S3_ENDPOINT_URL", "S3_BUCKET", "S3_ARTIFACT_KEY",
        "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION",
        "POSTGRES_HOST", "POSTGRES_PORT", "POSTGRES_DB", "POSTGRES_USER",
        "POSTGRES_PASSWORD", "POSTGRES_SSLMODE",
    ]:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
except Exception:
    pass

required = [
    "S3_ENDPOINT_URL", "S3_BUCKET", "S3_ARTIFACT_KEY",
    "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION",
    "POSTGRES_HOST", "POSTGRES_DB", "POSTGRES_USER", "POSTGRES_PASSWORD",
]

missing = [k for k in required if not os.getenv(k)]
if missing:
    print("Missing env vars:", missing)
    print("Заполни их через Colab Secrets или os.environ в приватной ячейке.")
else:
    print("All required env vars are set.")


All required env vars are set.


## 3. S3: скачать текущие артефакты

Архив должен распаковаться flat-структурой:

- `corpus.joblib`
- `corpus_emb.npy`
- `meta.json`
- `retriever_model/...`
- `train_df.parquet`, `val_df.parquet`, `pairs_df.parquet` если они есть


In [ ]:
def get_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.environ["S3_ENDPOINT_URL"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        region_name=os.getenv("AWS_DEFAULT_REGION", "eu-west-4"),
        config=Config(signature_version="s3v4"),
    )


def download_artifacts_from_s3(out_dir: Path = ARTIFACTS_DIR) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = WORK_DIR / "artifacts_current.zip"

    s3 = get_s3_client()
    s3.download_file(
        os.environ["S3_BUCKET"],
        os.environ["S3_ARTIFACT_KEY"],
        str(zip_path),
    )

    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

    required_files = [
        out_dir / "corpus.joblib",
        out_dir / "corpus_emb.npy",
        out_dir / "meta.json",
        out_dir / "retriever_model" / "config.json",
        out_dir / "retriever_model" / "config_sentence_transformers.json",
    ]
    missing = [str(p) for p in required_files if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Artifacts are not flat or incomplete. Missing: {missing}")

    return out_dir

ARTIFACTS_DIR = download_artifacts_from_s3()
print("Artifacts extracted to:", ARTIFACTS_DIR)
print("Files:", [p.name for p in ARTIFACTS_DIR.iterdir()])


Artifacts extracted to: rag_retriever_v2_work/artifacts_current
Files: ['corpus.joblib', 'pairs_df.parquet', 'corpus_emb.npy', 'train_df.parquet', 'meta.json', 'val_df.parquet', 'retriever_model']


## 4. Загрузка текущего retriever и корпуса


In [ ]:
corpus = joblib.load(ARTIFACTS_DIR / "corpus.joblib")
corpus_emb = np.load(ARTIFACTS_DIR / "corpus_emb.npy")

retriever_model = SentenceTransformer(
    str(ARTIFACTS_DIR / "retriever_model"),
    device=DEVICE,
)
retriever_model.to(DEVICE)

print("Retriever model device:", DEVICE)
print("Corpus documents:", len(corpus))
print("Embeddings shape:", corpus_emb.shape)
print("Sources:")
display(pd.Series([d.get("source", "unknown") for d in corpus]).value_counts())

with open(ARTIFACTS_DIR / "meta.json", "r", encoding="utf-8") as f:
    old_meta = json.load(f)

old_meta


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retriever model device: cuda
Corpus documents: 9796
Embeddings shape: (9796, 768)
Sources:


,count
codesearchnet,3687
spark_docs,3218
hive_docs,1825
trino_docs,1056
neon_schema,10


{'run_id': '20260225_094607',
 'created_utc': '2026-02-25T09:46:14.134893Z',
 'llm_name': 'google/flan-t5-base',
 'emb_model_name': 'sentence-transformers/all-mpnet-base-v2',
 'chunk_size': 900,
 'chunk_overlap': 150,
 'seed': 42,
 'val_recall_at_10': 0.9909365558912386,
 'notes': 'Baseline RAG artifacts: retriever + corpus + corpus_emb (+pairs).'}

## 5. Подключение к Postgres и выгрузка database schema


In [ ]:
def get_pg_connection():
    return psycopg2.connect(
        host=os.environ["POSTGRES_HOST"],
        port=int(os.getenv("POSTGRES_PORT", "5432")),
        dbname=os.environ["POSTGRES_DB"],
        user=os.environ["POSTGRES_USER"],
        password=os.environ["POSTGRES_PASSWORD"],
        sslmode=os.getenv("POSTGRES_SSLMODE", "require"),
    )


def read_sql_df(query: str, params=None) -> pd.DataFrame:
    with get_pg_connection() as conn:
        return pd.read_sql(query, conn, params=params)

with get_pg_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print(cur.fetchone()[0])


PostgreSQL 17.8 (92d3c18) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


In [ ]:
tables_df = read_sql_df(
    """
    SELECT
        t.table_schema,
        t.table_name,
        obj_description(
            ('"' || t.table_schema || '"."' || t.table_name || '"')::regclass
        ) AS table_comment
    FROM information_schema.tables t
    WHERE t.table_schema = %s
      AND t.table_type = 'BASE TABLE'
    ORDER BY t.table_name;
    """,
    params=(SCHEMA,),
)

columns_df = read_sql_df(
    """
    SELECT
        c.table_schema,
        c.table_name,
        c.ordinal_position,
        c.column_name,
        c.data_type,
        c.udt_name,
        c.is_nullable,
        c.column_default,
        col_description(
            ('"' || c.table_schema || '"."' || c.table_name || '"')::regclass,
            c.ordinal_position
        ) AS column_comment
    FROM information_schema.columns c
    WHERE c.table_schema = %s
    ORDER BY c.table_name, c.ordinal_position;
    """,
    params=(SCHEMA,),
)

pk_df = read_sql_df(
    """
    SELECT
        tc.table_schema,
        tc.table_name,
        kcu.column_name,
        tc.constraint_name
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
       AND tc.table_schema = kcu.table_schema
    WHERE tc.constraint_type = 'PRIMARY KEY'
      AND tc.table_schema = %s
    ORDER BY tc.table_name, kcu.ordinal_position;
    """,
    params=(SCHEMA,),
)

fk_df = read_sql_df(
    """
    SELECT
        tc.table_schema,
        tc.table_name,
        kcu.column_name,
        ccu.table_schema AS foreign_table_schema,
        ccu.table_name AS foreign_table_name,
        ccu.column_name AS foreign_column_name,
        tc.constraint_name
    FROM information_schema.table_constraints AS tc
    JOIN information_schema.key_column_usage AS kcu
        ON tc.constraint_name = kcu.constraint_name
       AND tc.table_schema = kcu.table_schema
    JOIN information_schema.constraint_column_usage AS ccu
        ON ccu.constraint_name = tc.constraint_name
       AND ccu.table_schema = tc.table_schema
    WHERE tc.constraint_type = 'FOREIGN KEY'
      AND tc.table_schema = %s
    ORDER BY tc.table_name, kcu.column_name;
    """,
    params=(SCHEMA,),
)

display(tables_df)
display(columns_df)
display(pk_df)
display(fk_df)


/tmp/ipykernel_18891/2442057618.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn, params=params)


,table_schema,table_name,table_comment
0,rag_kg,campaigns,Marketing campaigns dimension. Contains campai...
1,rag_kg,customers,Customer dimension. Contains customer profile ...
2,rag_kg,daily_fx_rates,Daily foreign exchange rates. Contains currenc...
3,rag_kg,order_items,Order line items fact table. Contains product-...
4,rag_kg,orders,Order header fact table. Contains one row per ...
5,rag_kg,products,Product dimension. Contains product catalog at...
6,rag_kg,refunds,Refunds fact table. Contains refund events lin...
7,rag_kg,stores,Store dimension. Contains sales channel and st...
8,rag_kg,support_tickets,Customer support tickets fact table. Contains ...
9,rag_kg,web_events,Web events fact table. Contains customer digit...


,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable,column_default,column_comment
0,rag_kg,campaigns,1,campaign_id,bigint,int8,NO,nextval('rag_kg.campaigns_campaign_id_seq'::re...,Unique campaign identifier. Primary key of the...
1,rag_kg,campaigns,2,campaign_name,text,text,NO,None,Human-readable marketing campaign name.
2,rag_kg,campaigns,3,channel,text,text,YES,None,"Marketing channel, for example email, ads, pus..."
3,rag_kg,campaigns,4,start_date,date,date,NO,None,Date when the marketing campaign starts.
4,rag_kg,campaigns,5,end_date,date,date,YES,None,Date when the marketing campaign ends.
...,...,...,...,...,...,...,...,...,...
74,rag_kg,web_events,4,session_id,text,text,YES,None,Client-side session identifier.
75,rag_kg,web_events,5,event_type,text,text,NO,None,"Event type, for example page_view, search, add..."
76,rag_kg,web_events,6,page_url,text,text,YES,None,URL of the page where the web event occurred.
77,rag_kg,web_events,7,referrer,text,text,YES,None,Referrer URL or traffic source that led the us...


,table_schema,table_name,column_name,constraint_name
0,rag_kg,campaigns,campaign_id,campaigns_pkey
1,rag_kg,customers,customer_id,customers_pkey
2,rag_kg,daily_fx_rates,rate_date,daily_fx_rates_pkey
3,rag_kg,daily_fx_rates,base_currency,daily_fx_rates_pkey
4,rag_kg,daily_fx_rates,quote_currency,daily_fx_rates_pkey
5,rag_kg,order_items,order_item_id,order_items_pkey
6,rag_kg,orders,order_id,orders_pkey
7,rag_kg,products,product_id,products_pkey
8,rag_kg,refunds,refund_id,refunds_pkey
9,rag_kg,stores,store_id,stores_pkey


,table_schema,table_name,column_name,foreign_table_schema,foreign_table_name,foreign_column_name,constraint_name
0,rag_kg,order_items,order_id,rag_kg,orders,order_id,order_items_order_id_fkey
1,rag_kg,order_items,product_id,rag_kg,products,product_id,order_items_product_id_fkey
2,rag_kg,orders,campaign_id,rag_kg,campaigns,campaign_id,orders_campaign_id_fkey
3,rag_kg,orders,customer_id,rag_kg,customers,customer_id,orders_customer_id_fkey
4,rag_kg,orders,store_id,rag_kg,stores,store_id,orders_store_id_fkey
5,rag_kg,refunds,order_id,rag_kg,orders,order_id,refunds_order_id_fkey
6,rag_kg,support_tickets,customer_id,rag_kg,customers,customer_id,support_tickets_customer_id_fkey
7,rag_kg,web_events,customer_id,rag_kg,customers,customer_id,web_events_customer_id_fkey


## 6. Контроль полноты описаний таблиц и колонок


In [ ]:
missing_table_comments = tables_df[
    tables_df["table_comment"].isna()
    | (tables_df["table_comment"].astype(str).str.strip() == "")
]

missing_column_comments = columns_df[
    columns_df["column_comment"].isna()
    | (columns_df["column_comment"].astype(str).str.strip() == "")
]

print("Tables:", len(tables_df))
print("Columns:", len(columns_df))
print("Missing table comments:", len(missing_table_comments))
print("Missing column comments:", len(missing_column_comments))

display(missing_table_comments)
display(missing_column_comments)

assert len(missing_table_comments) == 0, "Есть таблицы без описания"
assert len(missing_column_comments) == 0, "Есть колонки без описания"


Tables: 10
Columns: 79
Missing table comments: 0
Missing column comments: 0


,table_schema,table_name,table_comment


,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable,column_default,column_comment


## 7. Создание обогащенного `database_schema` corpus

Один документ = одна таблица.  
Чтобы SQL retrieval лучше находил несколько связанных таблиц, schema-документы дополнительно обогащаются:

- business terms / aliases;
- typical analytical use cases;
- common joins;
- SQL generation hints.


In [ ]:

TABLE_ENRICHMENT = {
    "orders": {
        "business_terms": [
            "orders", "sales", "revenue", "gross revenue", "order amount",
            "daily revenue", "monthly revenue", "payment method", "order status",
        ],
        "typical_questions": [
            "Show total revenue by day, week or month.",
            "Calculate revenue by customer segment.",
            "Show revenue by store type.",
            "Show campaign revenue by marketing channel.",
            "Count orders by status or payment method.",
            "Analyze refunded orders together with refunds.",
            "Convert order revenue between currencies using daily FX rates.",
        ],
        "sql_hints": [
            "Use total_amount for order-level revenue analysis.",
            "Join orders.customer_id to customers.customer_id for customer segmentation.",
            "Join orders.store_id to stores.store_id for store and channel analysis.",
            "Join orders.campaign_id to campaigns.campaign_id for campaign performance.",
            "Join orders.order_id to order_items.order_id for product-level sales.",
            "Join orders.order_id to refunds.order_id for refund analysis.",
            "Use order_ts for date, month and time-based aggregations.",
        ],
    },
    "customers": {
        "business_terms": [
            "customers", "users", "clients", "customer segment", "B2C", "B2B",
            "VIP", "customer country", "customer city", "active customers",
        ],
        "typical_questions": [
            "Show revenue by customer segment.",
            "Count customers by country or city.",
            "Analyze orders for active customers.",
            "Join customers with support tickets.",
            "Join customers with web events.",
        ],
        "sql_hints": [
            "Use segment for customer segmentation.",
            "Join customers.customer_id to orders.customer_id for revenue by segment.",
            "Join customers.customer_id to support_tickets.customer_id for support analytics.",
            "Join customers.customer_id to web_events.customer_id for digital behavior analytics.",
        ],
    },
    "order_items": {
        "business_terms": [
            "order items", "line items", "product sales", "quantity", "unit price",
            "gross sales", "discount", "tax", "basket", "cart",
        ],
        "typical_questions": [
            "Show sales by product category.",
            "Calculate gross sales by product.",
            "Analyze quantity sold by product or brand.",
            "Calculate discounts and taxes by category.",
        ],
        "sql_hints": [
            "Use quantity * unit_price - discount_amount for line-level sales.",
            "Join order_items.product_id to products.product_id for product attributes.",
            "Join order_items.order_id to orders.order_id for order dates and customers.",
        ],
    },
    "products": {
        "business_terms": [
            "products", "SKU", "catalog", "product category", "brand",
            "product sales", "category sales", "list price",
        ],
        "typical_questions": [
            "Show sales by product category.",
            "Show sales by brand.",
            "Find discontinued products.",
            "Analyze product catalog by category.",
        ],
        "sql_hints": [
            "Use category for category-level sales analysis.",
            "Use brand for brand-level grouping.",
            "Join products.product_id to order_items.product_id for sales by product.",
        ],
    },
    "stores": {
        "business_terms": [
            "stores", "sales channels", "store type", "online", "retail", "partner",
            "store country", "store city", "channel revenue",
        ],
        "typical_questions": [
            "Show revenue by store type.",
            "Show revenue by store country or city.",
            "Compare online, retail and partner channels.",
            "Count active stores by country.",
        ],
        "sql_hints": [
            "Use store_type for online/retail/partner channel analysis.",
            "Join stores.store_id to orders.store_id for revenue by store or channel.",
        ],
    },
    "campaigns": {
        "business_terms": [
            "campaigns", "marketing campaigns", "marketing channel", "ads", "email",
            "push", "campaign budget", "campaign objective", "campaign performance",
        ],
        "typical_questions": [
            "Show campaign revenue by marketing channel.",
            "Analyze campaign performance by objective.",
            "Compare campaign budget and generated revenue.",
            "Show orders by campaign.",
        ],
        "sql_hints": [
            "Use channel for marketing channel grouping.",
            "Use objective for acquisition, retention or reactivation analysis.",
            "Join campaigns.campaign_id to orders.campaign_id for campaign revenue.",
        ],
    },
    "refunds": {
        "business_terms": [
            "refunds", "returns", "refund amount", "refund reason", "refunded orders",
            "partial return", "product issue",
        ],
        "typical_questions": [
            "Calculate refund amount by reason.",
            "Show refunds by month.",
            "Analyze refunded orders.",
            "Calculate refund rate using orders and refunds.",
        ],
        "sql_hints": [
            "Use amount for refund amount analysis.",
            "Use refund_ts for time-based refund analysis.",
            "Join refunds.order_id to orders.order_id to analyze refunded orders and refund rate.",
        ],
    },
    "support_tickets": {
        "business_terms": [
            "support tickets", "customer support", "ticket priority", "ticket status",
            "support category", "closed tickets", "open tickets", "time to close",
        ],
        "typical_questions": [
            "Show support tickets by priority and status.",
            "Calculate average time to close support tickets.",
            "Show tickets by category.",
            "Join support tickets with customers.",
        ],
        "sql_hints": [
            "Use created_ts and closed_ts to calculate time to close.",
            "Use priority and status for support workload analysis.",
            "Join support_tickets.customer_id to customers.customer_id for customer-level support analytics.",
        ],
    },
    "web_events": {
        "business_terms": [
            "web events", "digital behavior", "session", "page view", "search",
            "add to cart", "purchase event", "conversion funnel", "referrer",
        ],
        "typical_questions": [
            "Show purchase events by session.",
            "Calculate conversion funnel from page views to purchases.",
            "Show page views by referrer.",
            "Analyze add-to-cart and purchase events.",
            "Join web events with customers.",
        ],
        "sql_hints": [
            "Use session_id for session-level funnel analysis.",
            "Use event_type for page_view, search, add_to_cart and purchase events.",
            "Join web_events.customer_id to customers.customer_id when customer is known.",
        ],
    },
    "daily_fx_rates": {
        "business_terms": [
            "FX rates", "currency conversion", "exchange rate", "EUR to USD",
            "USD to EUR", "base currency", "quote currency", "daily rate",
        ],
        "typical_questions": [
            "Convert daily order revenue from EUR to USD.",
            "Convert revenue using daily FX rates.",
            "Analyze currency conversion by date.",
        ],
        "sql_hints": [
            "Join rate_date to the date part of order_ts when converting order revenue.",
            "Use base_currency and quote_currency to select the conversion direction.",
            "Use rate to convert amount from base currency to quote currency.",
        ],
    },
}


def _format_list_block(title: str, values: list[str]) -> list[str]:
    if not values:
        return []
    lines = ["", f"{title}:"]
    for value in values:
        lines.append(f"- {value}")
    return lines


def build_schema_documents(
    tables_df: pd.DataFrame,
    columns_df: pd.DataFrame,
    pk_df: pd.DataFrame,
    fk_df: pd.DataFrame,
    schema: str,
) -> list[dict]:
    docs = []

    columns_by_table = {
        table_name: group.sort_values("ordinal_position")
        for table_name, group in columns_df.groupby("table_name")
    }

    pk_by_table = (
        {
            table_name: group["column_name"].tolist()
            for table_name, group in pk_df.groupby("table_name")
        }
        if len(pk_df)
        else {}
    )

    fk_by_table = (
        {
            table_name: group.to_dict(orient="records")
            for table_name, group in fk_df.groupby("table_name")
        }
        if len(fk_df)
        else {}
    )

    for _, table_row in tables_df.sort_values("table_name").iterrows():
        table_name = table_row["table_name"]
        table_comment = table_row["table_comment"]
        enrich = TABLE_ENRICHMENT.get(table_name, {})

        lines = [
            f"TABLE {schema}.{table_name}",
            f"Description: {table_comment}",
        ]

        lines += _format_list_block("Business terms and aliases", enrich.get("business_terms", []))
        lines += _format_list_block("Typical analytical questions", enrich.get("typical_questions", []))

        lines.append("")
        lines.append("Columns:")

        for _, col in columns_by_table[table_name].iterrows():
            nullable = "nullable" if col["is_nullable"] == "YES" else "not nullable"
            default = f", default={col['column_default']}" if pd.notna(col["column_default"]) else ""

            lines.append(
                f"- {table_name}.{col['column_name']} "
                f"({col['data_type']}, {nullable}{default}): "
                f"{col['column_comment']}"
            )

        pks = pk_by_table.get(table_name, [])
        if pks:
            lines.append("")
            lines.append("Primary key:")
            for pk in pks:
                lines.append(f"- {table_name}.{pk}")

        fks = fk_by_table.get(table_name, [])
        if fks:
            lines.append("")
            lines.append("Foreign keys:")
            for fk in fks:
                lines.append(
                    f"- {table_name}.{fk['column_name']} -> "
                    f"{fk['foreign_table_name']}.{fk['foreign_column_name']}"
                )

        reverse_fks = fk_df[fk_df["foreign_table_name"] == table_name] if len(fk_df) else pd.DataFrame()
        if len(reverse_fks):
            lines.append("")
            lines.append("Referenced by:")
            for _, fk in reverse_fks.iterrows():
                lines.append(
                    f"- {fk['table_name']}.{fk['column_name']} -> "
                    f"{table_name}.{fk['foreign_column_name']}"
                )

        lines += _format_list_block("SQL generation hints", enrich.get("sql_hints", []))

        # Этот блок помогает retriever на вопросах, где пользователь говорит бизнес-терминами.
        search_keywords = []
        search_keywords.extend(enrich.get("business_terms", []))
        search_keywords.extend(enrich.get("typical_questions", []))
        search_keywords.extend(enrich.get("sql_hints", []))
        if search_keywords:
            lines.append("")
            lines.append("Search keywords:")
            lines.append(" | ".join(search_keywords))

        text = "\n".join(lines)

        doc = {
            "doc_id": f"database_schema::{schema}.{table_name}",
            "source": "database_schema",
            "title": f"{schema}.{table_name}",
            "text": text,
            "metadata": {
                "schema": schema,
                "table": table_name,
                "columns": columns_by_table[table_name]["column_name"].tolist(),
                "primary_keys": pks,
                "foreign_keys": [
                    {
                        "column": fk["column_name"],
                        "foreign_table": fk["foreign_table_name"],
                        "foreign_column": fk["foreign_column_name"],
                    }
                    for fk in fks
                ],
                "business_terms": enrich.get("business_terms", []),
                "typical_questions": enrich.get("typical_questions", []),
            },
        }

        docs.append(doc)

    return docs


schema_docs = build_schema_documents(tables_df, columns_df, pk_df, fk_df, SCHEMA)

print("Schema docs:", len(schema_docs))
for doc in schema_docs:
    print("-", doc["doc_id"], "| chars:", len(doc["text"]))

print("\nPreview orders document:\n")
orders_doc = next(d for d in schema_docs if d["doc_id"] == f"database_schema::{SCHEMA}.orders")
print(orders_doc["text"][:3000])


Schema docs: 10
- database_schema::rag_kg.campaigns | chars: 2216
- database_schema::rag_kg.customers | chars: 2615
- database_schema::rag_kg.daily_fx_rates | chars: 1726
- database_schema::rag_kg.order_items | chars: 2243
- database_schema::rag_kg.orders | chars: 3530
- database_schema::rag_kg.products | chars: 1867
- database_schema::rag_kg.refunds | chars: 1822
- database_schema::rag_kg.stores | chars: 1940
- database_schema::rag_kg.support_tickets | chars: 2497
- database_schema::rag_kg.web_events | chars: 2353

Preview orders document:

TABLE rag_kg.orders
Description: Order header fact table. Contains one row per customer order with customer, store, campaign, timestamp, status, payment method, amount and currency.

Business terms and aliases:
- orders
- sales
- revenue
- gross revenue
- order amount
- daily revenue
- monthly revenue
- payment method
- order status

Typical analytical questions:
- Show total revenue by day, week or month.
- Calculate revenue by customer segment.
-

In [ ]:
schema_corpus_path = WORK_DIR / "database_schema_corpus.jsonl"
with schema_corpus_path.open("w", encoding="utf-8") as f:
    for doc in schema_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

schema_corpus_path


PosixPath('rag_retriever_v2_work/database_schema_corpus.jsonl')

## 8. Объединение корпусов

Добавляется новый `database_schema` корпус. Старые schema-документы удаляются.

In [ ]:
LEGACY_SCHEMA_SOURCES = {"database_schema", "neon_schema"}

base_corpus = [
    doc for doc in corpus
    if doc.get("source") not in LEGACY_SCHEMA_SOURCES
]
updated_corpus = base_corpus + schema_docs

print("Old corpus:", len(corpus))
print("Without legacy schema sources:", len(base_corpus))
print("New database_schema docs:", len(schema_docs))
print("Updated corpus:", len(updated_corpus))

source_counts = pd.Series([d.get("source", "unknown") for d in updated_corpus]).value_counts()
display(source_counts)

assert "neon_schema" not in set(source_counts.index), "Legacy source neon_schema is still present"
assert "database_schema" in set(source_counts.index), "database_schema docs were not added"
assert int(source_counts.loc["database_schema"]) == len(schema_docs), "Unexpected database_schema doc count"


Old corpus: 9796
Without legacy schema sources: 9786
New database_schema docs: 10
Updated corpus: 9796


,count
codesearchnet,3687
spark_docs,3218
hive_docs,1825
trino_docs,1056
database_schema,10


## 9. Подготовка train/eval пар

Для fine-tuning retriever используем пары `query -> positive_doc_id`.


In [ ]:
old_pairs_path = ARTIFACTS_DIR / "pairs_df.parquet"
if old_pairs_path.exists():
    old_pairs_df = pd.read_parquet(old_pairs_path)
    print("Loaded old pairs:", old_pairs_df.shape)
    display(old_pairs_df.head())
else:
    old_pairs_df = pd.DataFrame(columns=["pair_source", "query", "positive_doc_id"])
    print("No old pairs_df.parquet found")


Loaded old pairs: (3303, 3)


,pair_source,query,pos_doc_id
0,docs,https://spark.apache.org/docs/latest/,spark_docs::https://spark.apache.org/docs/late...
1,docs,index.html,spark_docs::index.html::0
2,docs,quick-start.html,spark_docs::quick-start.html::0
3,docs,rdd-programming-guide.html,spark_docs::rdd-programming-guide.html::0
4,docs,sql-programming-guide.html,spark_docs::sql-programming-guide.html::0


In [ ]:
def add_pair(rows: list[tuple[str, str, str]], query: str, *doc_ids: str, source: str = "sql_schema"):
    for doc_id in doc_ids:
        rows.append((source, query, doc_id))


sql_pairs: list[tuple[str, str, str]] = []

# Single-table schema questions
single_table_queries = {
    "database_schema::rag_kg.orders": [
        "Show total revenue by month",
        "Calculate daily revenue",
        "Count orders by status",
        "Show orders by payment method",
        "Find completed orders",
        "Analyze order lifecycle status",
        "Show total order amount by date",
    ],
    "database_schema::rag_kg.customers": [
        "Count customers by country",
        "Show customers by city",
        "List active customers",
        "Show customers by segment",
        "Analyze customer signup dates",
    ],
    "database_schema::rag_kg.products": [
        "Show products by category",
        "List products by brand",
        "Find discontinued products",
        "Analyze product catalog",
        "Show current list price by product",
    ],
    "database_schema::rag_kg.order_items": [
        "Show order line items",
        "Calculate quantity sold",
        "Calculate line item sales",
        "Analyze discounts by line item",
        "Analyze tax amount by line item",
    ],
    "database_schema::rag_kg.stores": [
        "Show stores by country",
        "Show stores by city",
        "Count stores by store type",
        "List active sales channels",
        "Analyze online retail and partner channels",
    ],
    "database_schema::rag_kg.campaigns": [
        "Show campaigns by marketing channel",
        "Analyze campaign budget",
        "Show campaigns by objective",
        "Compare acquisition and retention campaigns",
    ],
    "database_schema::rag_kg.refunds": [
        "Calculate refund amount by reason",
        "Show refunds by month",
        "Analyze refund reasons",
        "Show refund events",
    ],
    "database_schema::rag_kg.support_tickets": [
        "Show support tickets by priority and status",
        "Calculate average time to close support tickets",
        "Show tickets by category",
        "Analyze open and closed support tickets",
    ],
    "database_schema::rag_kg.web_events": [
        "Show purchase events by session",
        "Calculate conversion funnel from page views to purchases",
        "Show page views by referrer",
        "Analyze add to cart events",
        "Analyze user sessions",
    ],
    "database_schema::rag_kg.daily_fx_rates": [
        "Show daily FX rates",
        "Convert EUR to USD using exchange rates",
        "Analyze currency conversion rates",
        "Find exchange rate by date",
    ],
}

for doc_id, queries in single_table_queries.items():
    for query in queries:
        add_pair(sql_pairs, query, doc_id)

# Multi-table analytical questions
multi_table_queries = [
    (
        "Show revenue by customer segment",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.customers",
    ),
    (
        "Calculate order amount grouped by customer segment",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.customers",
    ),
    (
        "Show revenue by customer country",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.customers",
    ),
    (
        "Show sales by product category",
        "database_schema::rag_kg.order_items",
        "database_schema::rag_kg.products",
    ),
    (
        "Calculate gross sales by product category",
        "database_schema::rag_kg.order_items",
        "database_schema::rag_kg.products",
    ),
    (
        "Show quantity sold by brand",
        "database_schema::rag_kg.order_items",
        "database_schema::rag_kg.products",
    ),
    (
        "Join orders with order items and products to calculate product sales",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.order_items",
        "database_schema::rag_kg.products",
    ),
    (
        "Join orders with refunds to analyze refunded orders",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.refunds",
    ),
    (
        "Calculate refund rate by month",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.refunds",
    ),
    (
        "Show revenue by store type",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.stores",
    ),
    (
        "Compare order revenue across countries and stores",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.stores",
    ),
    (
        "Show revenue by sales channel",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.stores",
    ),
    (
        "Show campaign revenue by marketing channel",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.campaigns",
    ),
    (
        "Analyze campaign performance by objective",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.campaigns",
    ),
    (
        "Compare campaign budget and revenue",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.campaigns",
    ),
    (
        "Join support tickets with customers",
        "database_schema::rag_kg.support_tickets",
        "database_schema::rag_kg.customers",
    ),
    (
        "Show support tickets by customer segment",
        "database_schema::rag_kg.support_tickets",
        "database_schema::rag_kg.customers",
    ),
    (
        "Join web events with customers",
        "database_schema::rag_kg.web_events",
        "database_schema::rag_kg.customers",
    ),
    (
        "Show purchase events by customer segment",
        "database_schema::rag_kg.web_events",
        "database_schema::rag_kg.customers",
    ),
    (
        "Convert daily order revenue from EUR to USD",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.daily_fx_rates",
    ),
    (
        "Convert order amounts using daily FX rates",
        "database_schema::rag_kg.orders",
        "database_schema::rag_kg.daily_fx_rates",
    ),
]

for item in multi_table_queries:
    query, *doc_ids = item
    add_pair(sql_pairs, query, *doc_ids)

sql_pairs_df = pd.DataFrame(sql_pairs, columns=["pair_source", "query", "positive_doc_id"])
sql_pairs_df = sql_pairs_df.drop_duplicates(subset=["query", "positive_doc_id"]).reset_index(drop=True)

print("SQL synthetic pairs:", sql_pairs_df.shape)
display(sql_pairs_df.head(20))
display(sql_pairs_df["positive_doc_id"].value_counts())


SQL synthetic pairs: (91, 3)


,pair_source,query,positive_doc_id
0,sql_schema,Show total revenue by month,database_schema::rag_kg.orders
1,sql_schema,Calculate daily revenue,database_schema::rag_kg.orders
2,sql_schema,Count orders by status,database_schema::rag_kg.orders
3,sql_schema,Show orders by payment method,database_schema::rag_kg.orders
4,sql_schema,Find completed orders,database_schema::rag_kg.orders
5,sql_schema,Analyze order lifecycle status,database_schema::rag_kg.orders
6,sql_schema,Show total order amount by date,database_schema::rag_kg.orders
7,sql_schema,Count customers by country,database_schema::rag_kg.customers
8,sql_schema,Show customers by city,database_schema::rag_kg.customers
9,sql_schema,List active customers,database_schema::rag_kg.customers


,count
positive_doc_id,
database_schema::rag_kg.orders,21
database_schema::rag_kg.customers,12
database_schema::rag_kg.products,9
database_schema::rag_kg.order_items,9
database_schema::rag_kg.stores,8
database_schema::rag_kg.campaigns,7
database_schema::rag_kg.web_events,7
database_schema::rag_kg.refunds,6
database_schema::rag_kg.support_tickets,6


In [ ]:
def get_doc_id(doc: dict) -> str:
    return doc.get("doc_id") or f"{doc.get('source')}::{doc.get('title')}::{doc.get('chunk_id', 0)}"

updated_doc_ids = {get_doc_id(doc) for doc in updated_corpus}
missing_pos = sorted(set(sql_pairs_df["positive_doc_id"]) - updated_doc_ids)
print("Missing SQL positive doc ids:", missing_pos)
assert not missing_pos


Missing SQL positive doc ids: []


In [ ]:
all_pairs_df = pd.concat([old_pairs_df, sql_pairs_df], ignore_index=True)
all_pairs_df = all_pairs_df.dropna(subset=["query", "positive_doc_id"])
all_pairs_df = all_pairs_df.drop_duplicates(subset=["query", "positive_doc_id"])

updated_doc_ids = {get_doc_id(doc) for doc in updated_corpus}
all_pairs_df["positive_doc_exists"] = all_pairs_df["positive_doc_id"].isin(updated_doc_ids)
missing_pair_count = int((~all_pairs_df["positive_doc_exists"]).sum())

print("Pairs before filtering missing positives:", len(all_pairs_df))
print("Pairs with missing positive_doc_id:", missing_pair_count)

if missing_pair_count:
    display(all_pairs_df.loc[~all_pairs_df["positive_doc_exists"], ["pair_source", "query", "positive_doc_id"]].head(20))

all_pairs_df = all_pairs_df[all_pairs_df["positive_doc_exists"]].drop(columns=["positive_doc_exists"])
all_pairs_df = all_pairs_df.reset_index(drop=True)

print("Pairs after filtering:", all_pairs_df.shape)
display(all_pairs_df["pair_source"].value_counts())
display(all_pairs_df.sample(min(10, len(all_pairs_df)), random_state=SEED))


Pairs before filtering missing positives: 91
Pairs with missing positive_doc_id: 0
Pairs after filtering: (91, 4)


,count
pair_source,
sql_schema,91


,pair_source,query,pos_doc_id,positive_doc_id
40,sql_schema,Calculate conversion funnel from page views to...,NaN,database_schema::rag_kg.web_events
22,sql_schema,Show stores by country,NaN,database_schema::rag_kg.stores
55,sql_schema,Show sales by product category,NaN,database_schema::rag_kg.products
88,sql_schema,Convert daily order revenue from EUR to USD,NaN,database_schema::rag_kg.daily_fx_rates
0,sql_schema,Show total revenue by month,NaN,database_schema::rag_kg.orders
26,sql_schema,Analyze online retail and partner channels,NaN,database_schema::rag_kg.stores
39,sql_schema,Show purchase events by session,NaN,database_schema::rag_kg.web_events
66,sql_schema,Calculate refund rate by month,NaN,database_schema::rag_kg.refunds
10,sql_schema,Show customers by segment,NaN,database_schema::rag_kg.customers
44,sql_schema,Show daily FX rates,NaN,database_schema::rag_kg.daily_fx_rates


## 10. Retriever evaluation benchmark

Оцениваем не только Recall@10, но и теперь:

- `source_hit@k`
- `doc_recall@k`
- `doc_full_hit@k`
- `mrr@10`


In [ ]:

retrieval_eval = [
    # documentation
    {"id": "docs_001", "question": "What is Apache Spark?", "expected_sources": ["spark_docs"], "expected_doc_ids": []},
    {"id": "docs_002", "question": "How to launch a PySpark session?", "expected_sources": ["spark_docs"], "expected_doc_ids": []},
    {"id": "docs_003", "question": "What is Trino and what does data federation mean?", "expected_sources": ["trino_docs"], "expected_doc_ids": []},
    {"id": "docs_004", "question": "What is Hive Metastore and why is it important?", "expected_sources": ["hive_docs"], "expected_doc_ids": []},

    # code
    {"id": "code_001", "question": "How do I read a JSON file in Python?", "expected_sources": ["codesearchnet"], "expected_doc_ids": []},
    {"id": "code_002", "question": "How can I safely get a nested value from a Python dict?", "expected_sources": ["codesearchnet"], "expected_doc_ids": []},

    # database schema: single and multi-table
    {
        "id": "sql_001",
        "question": "Show revenue by customer segment.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.orders", "database_schema::rag_kg.customers"],
    },
    {
        "id": "sql_002",
        "question": "Show sales by product category.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.order_items", "database_schema::rag_kg.products"],
    },
    {
        "id": "sql_003",
        "question": "Calculate refund amount by refund reason.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.refunds"],
    },
    {
        "id": "sql_004",
        "question": "Show support tickets by priority and status.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.support_tickets"],
    },
    {
        "id": "sql_005",
        "question": "Show purchase events by session.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.web_events"],
    },
    {
        "id": "sql_006",
        "question": "Show campaign revenue by marketing channel.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.orders", "database_schema::rag_kg.campaigns"],
    },
    {
        "id": "sql_007",
        "question": "Show revenue by store type.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.orders", "database_schema::rag_kg.stores"],
    },
    {
        "id": "sql_008",
        "question": "Convert daily order revenue from EUR to USD.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.orders", "database_schema::rag_kg.daily_fx_rates"],
    },
    {
        "id": "sql_009",
        "question": "Calculate refund rate by month.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.orders", "database_schema::rag_kg.refunds"],
    },
    {
        "id": "sql_010",
        "question": "Show support tickets by customer segment.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.support_tickets", "database_schema::rag_kg.customers"],
    },
    {
        "id": "sql_011",
        "question": "Show quantity sold by brand.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.order_items", "database_schema::rag_kg.products"],
    },
    {
        "id": "sql_012",
        "question": "Show revenue by sales channel.",
        "expected_sources": ["database_schema"],
        "expected_doc_ids": ["database_schema::rag_kg.orders", "database_schema::rag_kg.stores"],
    },
]

retrieval_eval_df = pd.DataFrame(retrieval_eval)
retrieval_eval_df


,id,question,expected_sources,expected_doc_ids
0,docs_001,What is Apache Spark?,[spark_docs],[]
1,docs_002,How to launch a PySpark session?,[spark_docs],[]
2,docs_003,What is Trino and what does data federation mean?,[trino_docs],[]
3,docs_004,What is Hive Metastore and why is it important?,[hive_docs],[]
4,code_001,How do I read a JSON file in Python?,[codesearchnet],[]
5,code_002,How can I safely get a nested value from a Pyt...,[codesearchnet],[]
6,sql_001,Show revenue by customer segment.,[database_schema],"[database_schema::rag_kg.orders, database_sche..."
7,sql_002,Show sales by product category.,[database_schema],"[database_schema::rag_kg.order_items, database..."
8,sql_003,Calculate refund amount by refund reason.,[database_schema],[database_schema::rag_kg.refunds]
9,sql_004,Show support tickets by priority and status.,[database_schema],[database_schema::rag_kg.support_tickets]


### 10.1 Baseline evaluation: текущая модель + обновлённый corpus без fine-tuning


In [ ]:
def doc_to_retrieval_text(doc: dict) -> str:
    source = doc.get("source", "")
    title = doc.get("title", "")
    text = doc.get("text", "")
    return f"Source: {source}\nTitle: {title}\n{text}"


@torch.inference_mode()
def compute_embeddings(
    model: SentenceTransformer,
    docs: list[dict],
    batch_size: int | None = None,
    device: str = DEVICE,
) -> np.ndarray:
    """Encode documents on GPU when available and return normalized float32 numpy matrix."""
    model.to(device)
    texts = [doc_to_retrieval_text(doc) for doc in docs]

    if batch_size is None:
        batch_size = DEFAULT_EMBED_BATCH_SIZE

    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
    )

    if device == "cuda":
        torch.cuda.empty_cache()

    return np.asarray(emb, dtype="float32")


updated_emb_baseline = compute_embeddings(
    retriever_model,
    updated_corpus,
    batch_size=DEFAULT_EMBED_BATCH_SIZE,
    device=DEVICE,
)
updated_emb_baseline.shape


Batches:   0%|          | 0/77 [00:00<?, ?it/s]

(9796, 768)

In [ ]:

_EMB_TENSOR_CACHE = {}


def get_embedding_tensor(emb: np.ndarray, device: str = DEVICE) -> torch.Tensor:
    """Cache embedding matrix on GPU/CPU for fast repeated retrieval evaluation."""
    key = (id(emb), device)
    if key not in _EMB_TENSOR_CACHE:
        _EMB_TENSOR_CACHE[key] = torch.as_tensor(emb, dtype=torch.float32, device=device)
    return _EMB_TENSOR_CACHE[key]


def filter_docs_and_embeddings(
    docs: list[dict],
    emb: np.ndarray,
    source_filter: set[str] | None = None,
) -> tuple[list[dict], np.ndarray]:
    if not source_filter:
        return docs, emb

    indices = [
        idx for idx, doc in enumerate(docs)
        if doc.get("source") in source_filter
    ]
    filtered_docs = [docs[idx] for idx in indices]
    filtered_emb = emb[indices]
    return filtered_docs, filtered_emb


@torch.inference_mode()
def retrieve_with_embeddings(
    model: SentenceTransformer,
    docs: list[dict],
    emb: np.ndarray,
    query: str,
    top_k: int = 10,
    device: str = DEVICE,
    source_filter: set[str] | None = None,
) -> list[dict]:
    model.to(device)

    docs_for_search, emb_for_search = filter_docs_and_embeddings(
        docs=docs,
        emb=emb,
        source_filter=source_filter,
    )

    q_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
    )
    q_emb = torch.as_tensor(q_emb[0], dtype=torch.float32, device=device)

    emb_tensor = get_embedding_tensor(emb_for_search, device=device)
    scores_tensor = emb_tensor @ q_emb
    top_scores, top_idx = torch.topk(scores_tensor, k=min(top_k, len(docs_for_search)))

    top_scores = top_scores.detach().cpu().numpy()
    top_idx = top_idx.detach().cpu().numpy()

    out = []
    for rank, (idx, score) in enumerate(zip(top_idx, top_scores), start=1):
        doc = docs_for_search[int(idx)]
        out.append({
            "rank": rank,
            "score": float(score),
            "doc_id": get_doc_id(doc),
            "source": doc.get("source"),
            "title": doc.get("title"),
            "text_preview": doc.get("text", "")[:300],
        })
    return out


def evaluate_retriever(
    eval_rows: list[dict],
    model: SentenceTransformer,
    docs: list[dict],
    emb: np.ndarray,
    top_k_values=(1, 3, 5, 10),
    device: str = DEVICE,
    source_filter: set[str] | None = None,
    eval_name: str = "Retriever evaluation",
) -> pd.DataFrame:
    records = []

    for item in tqdm(eval_rows, desc=eval_name):
        max_k = max(top_k_values)
        results = retrieve_with_embeddings(
            model=model,
            docs=docs,
            emb=emb,
            query=item["question"],
            top_k=max_k,
            device=device,
            source_filter=source_filter,
        )

        result_doc_ids = [r["doc_id"] for r in results]
        result_sources = [r["source"] for r in results]

        expected_sources = set(item.get("expected_sources", []))
        expected_doc_ids = set(item.get("expected_doc_ids", []))

        row = {
            "id": item["id"],
            "question": item["question"],
            "expected_sources": list(expected_sources),
            "expected_doc_ids": list(expected_doc_ids),
            "top1_source": result_sources[0] if result_sources else None,
            "top1_doc_id": result_doc_ids[0] if result_doc_ids else None,
            "top1_score": results[0]["score"] if results else None,
        }

        mrr = 0.0
        if expected_doc_ids:
            for rank, doc_id in enumerate(result_doc_ids, start=1):
                if doc_id in expected_doc_ids:
                    mrr = 1.0 / rank
                    break
        row["mrr@10"] = mrr if expected_doc_ids else None

        for k in top_k_values:
            top_sources = set(result_sources[:k])
            top_doc_ids = set(result_doc_ids[:k])

            row[f"source_hit@{k}"] = int(bool(expected_sources & top_sources))

            if expected_doc_ids:
                row[f"doc_recall@{k}"] = len(expected_doc_ids & top_doc_ids) / len(expected_doc_ids)
                row[f"doc_full_hit@{k}"] = int(expected_doc_ids.issubset(top_doc_ids))
            else:
                row[f"doc_recall@{k}"] = None
                row[f"doc_full_hit@{k}"] = None

        records.append(row)

    return pd.DataFrame(records)


def get_sql_eval_rows(eval_rows: list[dict]) -> list[dict]:
    return [row for row in eval_rows if row["id"].startswith("sql_")]


def summarize_eval(df: pd.DataFrame) -> pd.Series:
    metric_cols = [
        c for c in df.columns
        if c.startswith("source_hit@")
        or c.startswith("doc_recall@")
        or c.startswith("doc_full_hit@")
        or c == "mrr@10"
    ]
    return df[metric_cols].mean(numeric_only=True).sort_index()


In [ ]:

baseline_eval_results = evaluate_retriever(
    retrieval_eval,
    retriever_model,
    updated_corpus,
    updated_emb_baseline,
    top_k_values=TOP_K_VALUES,
    device=DEVICE,
    source_filter=None,
    eval_name="Baseline global retrieval",
)

baseline_summary = summarize_eval(baseline_eval_results)

sql_eval_rows = get_sql_eval_rows(retrieval_eval)

baseline_schema_only_eval_results = evaluate_retriever(
    sql_eval_rows,
    retriever_model,
    updated_corpus,
    updated_emb_baseline,
    top_k_values=TOP_K_VALUES,
    device=DEVICE,
    source_filter={"database_schema"},
    eval_name="Baseline schema-only retrieval",
)

baseline_schema_only_summary = summarize_eval(baseline_schema_only_eval_results)

print("Baseline global summary")
display(baseline_summary)

print("Baseline schema-only SQL summary")
display(baseline_schema_only_summary)

print("SQL cases where not all expected tables are found in top-5, global mode:")
display(
    baseline_eval_results[
        baseline_eval_results["id"].str.startswith("sql_")
        & (baseline_eval_results["doc_full_hit@5"] == 0)
    ][["id", "question", "top1_source", "top1_doc_id", "doc_recall@5", "doc_full_hit@5", "doc_recall@10", "doc_full_hit@10"]]
)

print("SQL cases where not all expected tables are found in top-5, schema-only mode:")
display(
    baseline_schema_only_eval_results[
        baseline_schema_only_eval_results["doc_full_hit@5"] == 0
    ][["id", "question", "top1_source", "top1_doc_id", "doc_recall@5", "doc_full_hit@5", "doc_recall@10", "doc_full_hit@10"]]
)


Baseline global retrieval:   0%|          | 0/18 [00:00<?, ?it/s]

Baseline schema-only retrieval:   0%|          | 0/12 [00:00<?, ?it/s]

Baseline global summary


,0
doc_full_hit@1,0.166667
doc_full_hit@10,0.916667
doc_full_hit@3,0.666667
doc_full_hit@5,0.916667
doc_recall@1,0.541667
doc_recall@10,0.958333
doc_recall@3,0.833333
doc_recall@5,0.958333
mrr@10,0.958333
source_hit@1,0.944444


Baseline schema-only SQL summary


,0
doc_full_hit@1,0.250000
doc_full_hit@10,1.000000
doc_full_hit@3,0.833333
doc_full_hit@5,1.000000
doc_recall@1,0.625000
doc_recall@10,1.000000
doc_recall@3,0.916667
doc_recall@5,1.000000
mrr@10,1.000000
source_hit@1,1.000000


SQL cases where not all expected tables are found in top-5, global mode:


,id,question,top1_source,top1_doc_id,doc_recall@5,doc_full_hit@5,doc_recall@10,doc_full_hit@10
14,sql_009,Calculate refund rate by month.,database_schema,database_schema::rag_kg.refunds,0.5,0.0,0.5,0.0


SQL cases where not all expected tables are found in top-5, schema-only mode:


,id,question,top1_source,top1_doc_id,doc_recall@5,doc_full_hit@5,doc_recall@10,doc_full_hit@10


## 11. Fine-tuning retriever


In [ ]:
id_to_doc = {get_doc_id(doc): doc for doc in updated_corpus}

stratify_col = all_pairs_df["pair_source"] if all_pairs_df["pair_source"].nunique() > 1 else None
train_pairs_df, val_pairs_df = train_test_split(
    all_pairs_df,
    test_size=0.15,
    random_state=SEED,
    stratify=stratify_col,
)

print("Train:", train_pairs_df.shape)
print("Val:", val_pairs_df.shape)
display(train_pairs_df["pair_source"].value_counts())
display(val_pairs_df["pair_source"].value_counts())


Train: (77, 4)
Val: (14, 4)


,count
pair_source,
sql_schema,77


,count
pair_source,
sql_schema,14


In [ ]:
def make_examples(pairs_df: pd.DataFrame) -> list[InputExample]:
    examples = []
    skipped = 0
    for _, row in pairs_df.iterrows():
        pos_id = row["positive_doc_id"]
        doc = id_to_doc.get(pos_id)
        if not doc:
            skipped += 1
            continue
        examples.append(InputExample(texts=[row["query"], doc_to_retrieval_text(doc)]))
    print("Examples:", len(examples), "Skipped:", skipped)
    return examples

train_examples = make_examples(train_pairs_df)
val_examples = make_examples(val_pairs_df)


Examples: 77 Skipped: 0
Examples: 14 Skipped: 0


In [ ]:
DO_FINE_TUNING = True

TRAIN_BATCH_SIZE = TRAIN_BATCH_SIZE_DEFAULT
NUM_EPOCHS = 2
WARMUP_RATIO = 0.1
LEARNING_RATE = 2e-5

fine_tuned_model_dir = WORK_DIR / "retriever_model_v3"

if DO_FINE_TUNING:
    retriever_model.to(DEVICE)

    train_dataloader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=TRAIN_BATCH_SIZE,
        num_workers=DATALOADER_NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    train_loss = losses.MultipleNegativesRankingLoss(retriever_model)
    warmup_steps = int(len(train_dataloader) * NUM_EPOCHS * WARMUP_RATIO)

    fit_kwargs = dict(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=NUM_EPOCHS,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": LEARNING_RATE},
        output_path=str(fine_tuned_model_dir),
        show_progress_bar=True,
    )

    if USE_AMP:
        fit_kwargs["use_amp"] = True

    print("Fine-tuning config:")
    print("  DEVICE:", DEVICE)
    print("  TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)
    print("  NUM_EPOCHS:", NUM_EPOCHS)
    print("  WARMUP_STEPS:", warmup_steps)
    print("  LEARNING_RATE:", LEARNING_RATE)
    print("  USE_AMP:", USE_AMP)
    print("  DATALOADER_NUM_WORKERS:", DATALOADER_NUM_WORKERS)
    print("  train_examples:", len(train_examples))
    print("  val_examples:", len(val_examples))

    try:
        retriever_model.fit(**fit_kwargs)
    except TypeError:
        # На случай версии sentence-transformers без use_amp в fit()
        fit_kwargs.pop("use_amp", None)
        retriever_model.fit(**fit_kwargs)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    retriever_model_v2 = SentenceTransformer(str(fine_tuned_model_dir), device=DEVICE)
    retriever_model_v2.to(DEVICE)
else:
    print("Fine-tuning skipped. Reusing current retriever model.")
    retriever_model_v2 = retriever_model
    retriever_model_v2.to(DEVICE)


Fine-tuning config:
  DEVICE: cuda
  TRAIN_BATCH_SIZE: 32
  NUM_EPOCHS: 2
  WARMUP_STEPS: 0
  LEARNING_RATE: 2e-05
  USE_AMP: True
  DATALOADER_NUM_WORKERS: 2
  train_examples: 77
  val_examples: 14


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 12. Evaluation после fine-tuning / re-index


In [ ]:

_EMB_TENSOR_CACHE.clear()
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

updated_emb_v2 = compute_embeddings(
    retriever_model_v2,
    updated_corpus,
    batch_size=DEFAULT_EMBED_BATCH_SIZE,
    device=DEVICE,
)

v2_eval_results = evaluate_retriever(
    retrieval_eval,
    retriever_model_v2,
    updated_corpus,
    updated_emb_v2,
    top_k_values=TOP_K_VALUES,
    device=DEVICE,
    source_filter=None,
    eval_name="V2 global retrieval",
)

v2_summary = summarize_eval(v2_eval_results)

v2_schema_only_eval_results = evaluate_retriever(
    sql_eval_rows,
    retriever_model_v2,
    updated_corpus,
    updated_emb_v2,
    top_k_values=TOP_K_VALUES,
    device=DEVICE,
    source_filter={"database_schema"},
    eval_name="V2 schema-only retrieval",
)

v2_schema_only_summary = summarize_eval(v2_schema_only_eval_results)

comparison = pd.DataFrame({
    "baseline_global": baseline_summary,
    "baseline_schema_only_sql": baseline_schema_only_summary,
    "v2_global": v2_summary,
    "v2_schema_only_sql": v2_schema_only_summary,
})

comparison


Batches:   0%|          | 0/77 [00:00<?, ?it/s]

V2 global retrieval:   0%|          | 0/18 [00:00<?, ?it/s]

V2 schema-only retrieval:   0%|          | 0/12 [00:00<?, ?it/s]

,baseline_global,baseline_schema_only_sql,v2_global,v2_schema_only_sql
doc_full_hit@1,0.166667,0.250000,0.250000,0.250
doc_full_hit@10,0.916667,1.000000,1.000000,1.000
doc_full_hit@3,0.666667,0.833333,0.916667,1.000
doc_full_hit@5,0.916667,1.000000,1.000000,1.000
doc_recall@1,0.541667,0.625000,0.625000,0.625
doc_recall@10,0.958333,1.000000,1.000000,1.000
doc_recall@3,0.833333,0.916667,0.958333,1.000
doc_recall@5,0.958333,1.000000,1.000000,1.000
mrr@10,0.958333,1.000000,1.000000,1.000
source_hit@1,0.944444,1.000000,1.000000,1.000


In [ ]:

print("V2 global eval")
display(v2_eval_results)

print("V2 schema-only SQL eval")
display(v2_schema_only_eval_results)

print("Global SQL failures at top-5 after v2:")
display(
    v2_eval_results[
        v2_eval_results["id"].str.startswith("sql_")
        & (v2_eval_results["doc_full_hit@5"] == 0)
    ][["id", "question", "top1_source", "top1_doc_id", "doc_recall@5", "doc_full_hit@5", "doc_recall@10", "doc_full_hit@10"]]
)

print("Schema-only SQL failures at top-5 after v2:")
display(
    v2_schema_only_eval_results[
        v2_schema_only_eval_results["doc_full_hit@5"] == 0
    ][["id", "question", "top1_source", "top1_doc_id", "doc_recall@5", "doc_full_hit@5", "doc_recall@10", "doc_full_hit@10"]]
)


V2 global eval


,id,question,expected_sources,expected_doc_ids,top1_source,top1_doc_id,top1_score,mrr@10,source_hit@1,doc_recall@1,doc_full_hit@1,source_hit@3,doc_recall@3,doc_full_hit@3,source_hit@5,doc_recall@5,doc_full_hit@5,source_hit@10,doc_recall@10,doc_full_hit@10
0,docs_001,What is Apache Spark?,[spark_docs],[],spark_docs,spark_docs::https://spark.apache.org/docs/late...,0.626692,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN
1,docs_002,How to launch a PySpark session?,[spark_docs],[],spark_docs,spark_docs::pyspark.SparkContext.html::1,0.632449,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN
2,docs_003,What is Trino and what does data federation mean?,[trino_docs],[],trino_docs,trino_docs::concepts.html::1,0.660468,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN
3,docs_004,What is Hive Metastore and why is it important?,[hive_docs],[],hive_docs,hive_docs::Metastore+TLP+Proposal::4,0.726771,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN
4,code_001,How do I read a JSON file in Python?,[codesearchnet],[],codesearchnet,codesearchnet::codesn_1532::0,0.463255,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN
5,code_002,How can I safely get a nested value from a Pyt...,[codesearchnet],[],codesearchnet,codesearchnet::codesn_1071::0,0.652167,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN,1,NaN,NaN
6,sql_001,Show revenue by customer segment.,[database_schema],"[database_schema::rag_kg.orders, database_sche...",database_schema,database_schema::rag_kg.customers,0.547209,1.0,1,0.5,0.0,1,1.0,1.0,1,1.0,1.0,1,1.0,1.0
7,sql_002,Show sales by product category.,[database_schema],"[database_schema::rag_kg.order_items, database...",database_schema,database_schema::rag_kg.products,0.428057,1.0,1,0.5,0.0,1,1.0,1.0,1,1.0,1.0,1,1.0,1.0
8,sql_003,Calculate refund amount by refund reason.,[database_schema],[database_schema::rag_kg.refunds],database_schema,database_schema::rag_kg.refunds,0.534382,1.0,1,1.0,1.0,1,1.0,1.0,1,1.0,1.0,1,1.0,1.0
9,sql_004,Show support tickets by priority and status.,[database_schema],[database_schema::rag_kg.support_tickets],database_schema,database_schema::rag_kg.support_tickets,0.559946,1.0,1,1.0,1.0,1,1.0,1.0,1,1.0,1.0,1,1.0,1.0


V2 schema-only SQL eval


,id,question,expected_sources,expected_doc_ids,top1_source,top1_doc_id,top1_score,mrr@10,source_hit@1,doc_recall@1,doc_full_hit@1,source_hit@3,doc_recall@3,doc_full_hit@3,source_hit@5,doc_recall@5,doc_full_hit@5,source_hit@10,doc_recall@10,doc_full_hit@10
0,sql_001,Show revenue by customer segment.,[database_schema],"[database_schema::rag_kg.orders, database_sche...",database_schema,database_schema::rag_kg.customers,0.547209,1.0,1,0.5,0,1,1.0,1,1,1.0,1,1,1.0,1
1,sql_002,Show sales by product category.,[database_schema],"[database_schema::rag_kg.order_items, database...",database_schema,database_schema::rag_kg.products,0.428057,1.0,1,0.5,0,1,1.0,1,1,1.0,1,1,1.0,1
2,sql_003,Calculate refund amount by refund reason.,[database_schema],[database_schema::rag_kg.refunds],database_schema,database_schema::rag_kg.refunds,0.534382,1.0,1,1.0,1,1,1.0,1,1,1.0,1,1,1.0,1
3,sql_004,Show support tickets by priority and status.,[database_schema],[database_schema::rag_kg.support_tickets],database_schema,database_schema::rag_kg.support_tickets,0.559946,1.0,1,1.0,1,1,1.0,1,1,1.0,1,1,1.0,1
4,sql_005,Show purchase events by session.,[database_schema],[database_schema::rag_kg.web_events],database_schema,database_schema::rag_kg.web_events,0.523037,1.0,1,1.0,1,1,1.0,1,1,1.0,1,1,1.0,1
5,sql_006,Show campaign revenue by marketing channel.,[database_schema],"[database_schema::rag_kg.campaigns, database_s...",database_schema,database_schema::rag_kg.campaigns,0.586802,1.0,1,0.5,0,1,1.0,1,1,1.0,1,1,1.0,1
6,sql_007,Show revenue by store type.,[database_schema],"[database_schema::rag_kg.stores, database_sche...",database_schema,database_schema::rag_kg.stores,0.474220,1.0,1,0.5,0,1,1.0,1,1,1.0,1,1,1.0,1
7,sql_008,Convert daily order revenue from EUR to USD.,[database_schema],"[database_schema::rag_kg.daily_fx_rates, datab...",database_schema,database_schema::rag_kg.daily_fx_rates,0.533489,1.0,1,0.5,0,1,1.0,1,1,1.0,1,1,1.0,1
8,sql_009,Calculate refund rate by month.,[database_schema],"[database_schema::rag_kg.refunds, database_sch...",database_schema,database_schema::rag_kg.refunds,0.448858,1.0,1,0.5,0,1,1.0,1,1,1.0,1,1,1.0,1
9,sql_010,Show support tickets by customer segment.,[database_schema],"[database_schema::rag_kg.support_tickets, data...",database_schema,database_schema::rag_kg.support_tickets,0.556731,1.0,1,0.5,0,1,1.0,1,1,1.0,1,1,1.0,1


Global SQL failures at top-5 after v2:


,id,question,top1_source,top1_doc_id,doc_recall@5,doc_full_hit@5,doc_recall@10,doc_full_hit@10


Schema-only SQL failures at top-5 after v2:


,id,question,top1_source,top1_doc_id,doc_recall@5,doc_full_hit@5,doc_recall@10,doc_full_hit@10


In [ ]:

diff = np.abs(updated_emb_baseline - updated_emb_v2).mean()
max_diff = np.abs(updated_emb_baseline - updated_emb_v2).max()

print("mean abs diff:", diff)
print("max abs diff:", max_diff)

def export_score(summary: pd.Series) -> float:
    keys = ["doc_full_hit@5", "doc_recall@5", "mrr@10"]
    return float(sum(summary.get(k, 0.0) for k in keys))

baseline_export_score = export_score(baseline_schema_only_summary)
v2_export_score = export_score(v2_schema_only_summary)

USE_FINE_TUNED_FOR_EXPORT = bool(DO_FINE_TUNING and v2_export_score >= baseline_export_score)

print("baseline_schema_only_score:", baseline_export_score)
print("v2_schema_only_score:", v2_export_score)
print("USE_FINE_TUNED_FOR_EXPORT:", USE_FINE_TUNED_FOR_EXPORT)

export_model = retriever_model_v2 if USE_FINE_TUNED_FOR_EXPORT else retriever_model
export_emb = updated_emb_v2 if USE_FINE_TUNED_FOR_EXPORT else updated_emb_baseline
export_summary = v2_summary if USE_FINE_TUNED_FOR_EXPORT else baseline_summary
export_schema_only_summary = v2_schema_only_summary if USE_FINE_TUNED_FOR_EXPORT else baseline_schema_only_summary

print("Export mode:", "fine-tuned model" if USE_FINE_TUNED_FOR_EXPORT else "re-indexed enriched corpus with original model")


mean abs diff: 0.0046990234
max abs diff: 0.07262564
baseline_schema_only_score: 3.0
v2_schema_only_score: 3.0
USE_FINE_TUNED_FOR_EXPORT: True
Export mode: fine-tuned model


## 12.1 Pre-export cleanup and validation

Перед сохранением артефактов ещё раз проверяем, что в корпусе нет старого `neon_schema`, а embeddings соответствуют очищенному корпусу.


In [ ]:
from collections import Counter

pre_export_counts = Counter(doc.get("source", "unknown") for doc in updated_corpus)
print("Pre-export source counts before defensive cleanup:")
print(pre_export_counts)

if pre_export_counts.get("neon_schema", 0) > 0:
    print("Removing legacy neon_schema documents before export...")
    updated_corpus = [doc for doc in updated_corpus if doc.get("source") != "neon_schema"]

post_export_counts = Counter(doc.get("source", "unknown") for doc in updated_corpus)
print("Pre-export source counts after cleanup:")
print(post_export_counts)

assert post_export_counts.get("neon_schema", 0) == 0, "neon_schema must not be exported"
assert post_export_counts.get("database_schema", 0) == len(schema_docs), "database_schema count mismatch"

_EMB_TENSOR_CACHE.clear()
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

export_emb = compute_embeddings(
    export_model,
    updated_corpus,
    batch_size=DEFAULT_EMBED_BATCH_SIZE,
    device=DEVICE,
)

print("Final corpus size:", len(updated_corpus))
print("Final embedding shape:", export_emb.shape)
assert export_emb.shape[0] == len(updated_corpus), "embeddings count != corpus size"

final_sources = sorted(set(doc.get("source", "unknown") for doc in updated_corpus))
print("Final sources:", final_sources)
assert "database_schema" in final_sources
assert "neon_schema" not in final_sources


Pre-export source counts before defensive cleanup:
Counter({'codesearchnet': 3687, 'spark_docs': 3218, 'hive_docs': 1825, 'trino_docs': 1056, 'database_schema': 10})
Pre-export source counts after cleanup:
Counter({'codesearchnet': 3687, 'spark_docs': 3218, 'hive_docs': 1825, 'trino_docs': 1056, 'database_schema': 10})


Batches:   0%|          | 0/77 [00:00<?, ?it/s]

Final corpus size: 9796
Final embedding shape: (9796, 768)
Final sources: ['codesearchnet', 'database_schema', 'hive_docs', 'spark_docs', 'trino_docs']


## 13. Сохранение новых артефактов


In [ ]:
if NEW_ARTIFACTS_DIR.exists():
    shutil.rmtree(NEW_ARTIFACTS_DIR)
NEW_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(updated_corpus, NEW_ARTIFACTS_DIR / "corpus.joblib")
np.save(NEW_ARTIFACTS_DIR / "corpus_emb.npy", export_emb.astype("float32"))

export_model.save(str(NEW_ARTIFACTS_DIR / "retriever_model"))

all_pairs_df.to_parquet(NEW_ARTIFACTS_DIR / "pairs_df.parquet", index=False)
train_pairs_df.to_parquet(NEW_ARTIFACTS_DIR / "train_df.parquet", index=False)
val_pairs_df.to_parquet(NEW_ARTIFACTS_DIR / "val_df.parquet", index=False)

final_source_counts = Counter(doc.get("source", "unknown") for doc in updated_corpus)

meta = {
    "version": "rag_retriever_v3_docs_code_database_schema_finetuned_clean",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "num_documents": len(updated_corpus),
    "embedding_shape": list(export_emb.shape),
    "sources": sorted(final_source_counts.keys()),
    "source_counts": dict(final_source_counts),
    "old_meta": old_meta,
    "evaluation": {
        "baseline_global": baseline_summary.to_dict(),
        "baseline_schema_only_sql": baseline_schema_only_summary.to_dict(),
        "v2_global": v2_summary.to_dict(),
        "v2_schema_only_sql": v2_schema_only_summary.to_dict(),
        "export_global": export_summary.to_dict(),
        "export_schema_only_sql": export_schema_only_summary.to_dict(),
        "comparison": comparison.to_dict(),
    },
    "export": {
        "use_fine_tuned_model": USE_FINE_TUNED_FOR_EXPORT,
        "mode": "fine_tuned" if USE_FINE_TUNED_FOR_EXPORT else "enriched_reindex",
        "mean_abs_embedding_diff": float(diff),
        "max_abs_embedding_diff": float(max_diff),
        "removed_legacy_sources": ["neon_schema"],
    },
    "runtime": {
        "device": DEVICE,
        "gpu_name": GPU_NAME,
        "use_amp": USE_AMP,
        "embed_batch_size": DEFAULT_EMBED_BATCH_SIZE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "recommended_sql_source_filter": "database_schema",
        "recommended_sql_top_k": 10,
        "recommended_docs_code_top_k": 5,
    },
    "notes": (
        "Clean retriever artifact with enriched database_schema corpus generated from Postgres comments. "
        "Legacy neon_schema documents were removed before export. "
        "Schema documents include business terms, typical analytical questions and SQL generation hints. "
        "For SQL route in runtime service use source_filter={'database_schema'} and top_k_schema=10."
    ),
}

(NEW_ARTIFACTS_DIR / "meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("Saved new artifacts to:", NEW_ARTIFACTS_DIR)
print([p.name for p in NEW_ARTIFACTS_DIR.iterdir()])
print("Export mode:", meta["export"]["mode"])
print("Final source counts:", final_source_counts)

assert "neon_schema" not in final_source_counts
assert (NEW_ARTIFACTS_DIR / "corpus.joblib").exists()
assert (NEW_ARTIFACTS_DIR / "corpus_emb.npy").exists()
assert (NEW_ARTIFACTS_DIR / "meta.json").exists()
assert (NEW_ARTIFACTS_DIR / "retriever_model" / "config.json").exists()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new artifacts to: rag_retriever_v2_work/artifacts_rag_baseline_latest
['corpus.joblib', 'pairs_df.parquet', 'corpus_emb.npy', 'train_df.parquet', 'meta.json', 'val_df.parquet', 'retriever_model']
Export mode: fine_tuned
Final source counts: Counter({'codesearchnet': 3687, 'spark_docs': 3218, 'hive_docs': 1825, 'trino_docs': 1056, 'database_schema': 10})


## 14. Формирование zip archive для S3


In [ ]:
zip_path = WORK_DIR / "artifacts_rag_baseline_latest.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in NEW_ARTIFACTS_DIR.rglob("*"):
        if not path.is_file():
            continue
        if path.name in {".DS_Store"}:
            continue
        if "__MACOSX" in path.parts or ".ipynb_checkpoints" in path.parts:
            continue
        arcname = path.relative_to(NEW_ARTIFACTS_DIR)
        zf.write(path, arcname)

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    print("First archive entries:", names[:30])
    assert "corpus.joblib" in names
    assert "corpus_emb.npy" in names
    assert "meta.json" in names
    assert "pairs_df.parquet" in names
    assert "train_df.parquet" in names
    assert "val_df.parquet" in names
    assert any(name.startswith("retriever_model/") for name in names)
    assert not any(name.startswith("artifacts_rag_baseline_latest/") for name in names)
    assert not any("__MACOSX" in name for name in names)
    assert not any(name.endswith(".DS_Store") for name in names)

print("Archive OK:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))


First archive entries: ['corpus.joblib', 'pairs_df.parquet', 'corpus_emb.npy', 'train_df.parquet', 'meta.json', 'val_df.parquet', 'retriever_model/README.md', 'retriever_model/model.safetensors', 'retriever_model/modules.json', 'retriever_model/config.json', 'retriever_model/config_sentence_transformers.json', 'retriever_model/tokenizer_config.json', 'retriever_model/sentence_bert_config.json', 'retriever_model/tokenizer.json', 'retriever_model/1_Pooling/config.json']
Archive OK: rag_retriever_v2_work/artifacts_rag_baseline_latest.zip
Size MB: 415.37


## 15. Upload clean archive в S3


In [ ]:
from collections import Counter
import joblib
import numpy as np

ARTIFACTS_DIR_CHECK = NEW_ARTIFACTS_DIR

corpus_check = joblib.load(ARTIFACTS_DIR_CHECK / "corpus.joblib")
emb_check = np.load(ARTIFACTS_DIR_CHECK / "corpus_emb.npy")
meta_check = json.loads((ARTIFACTS_DIR_CHECK / "meta.json").read_text(encoding="utf-8"))

source_counts_check = Counter(doc.get("source", "unknown") for doc in corpus_check)
print("Source counts:", source_counts_check)
print("Corpus size:", len(corpus_check))
print("Embedding shape:", emb_check.shape)
print("Meta sources:", meta_check.get("sources"))

assert "neon_schema" not in source_counts_check, "neon_schema found in final corpus"
assert "database_schema" in source_counts_check, "database_schema missing in final corpus"
assert emb_check.shape[0] == len(corpus_check), "embeddings count != corpus size"
assert meta_check["num_documents"] == len(corpus_check), "meta num_documents mismatch"
assert meta_check["embedding_shape"][0] == len(corpus_check), "meta embedding_shape mismatch"

print("Final artifact sanity check passed.")


Source counts: Counter({'codesearchnet': 3687, 'spark_docs': 3218, 'hive_docs': 1825, 'trino_docs': 1056, 'database_schema': 10})
Corpus size: 9796
Embedding shape: (9796, 768)
Meta sources: ['codesearchnet', 'database_schema', 'hive_docs', 'spark_docs', 'trino_docs']
Final artifact sanity check passed.


In [ ]:
UPLOAD_TO_S3 = True
TARGET_KEY = os.environ["S3_ARTIFACT_KEY"]

if UPLOAD_TO_S3:
    s3 = get_s3_client()
    s3.upload_file(
        str(zip_path),
        os.environ["S3_BUCKET"],
        TARGET_KEY,
    )
    print("Uploaded to:", f"s3://{os.environ['S3_BUCKET']}/{TARGET_KEY}")

    head = s3.head_object(
        Bucket=os.environ["S3_BUCKET"],
        Key=TARGET_KEY,
    )
    print("S3 object size MB:", round(head["ContentLength"] / 1024 / 1024, 2))
    print("S3 LastModified:", head["LastModified"])
    print("S3 ETag:", head.get("ETag"))
else:
    print("Upload skipped. Set UPLOAD_TO_S3=True when ready.")
    print("Archive path:", zip_path)


Uploaded to: s3://rag-assistant-storage/rag-baseline/artifacts_rag_baseline_latest.zip
S3 object size MB: 415.37
S3 LastModified: 2026-05-01 19:18:12+00:00
S3 ETag: "0632b41330a9923a0fc0b77c8d8fa211-52"


## 16. Smoke-test queries после обновления


In [ ]:
smoke_questions = [
    ("global", "What is Apache Spark?"),
    ("global", "How do I read a JSON file in Python?"),
    ("schema", "Show revenue by customer segment"),
    ("schema", "Show sales by product category"),
    ("schema", "Show campaign revenue by marketing channel"),
    ("schema", "Show revenue by store type"),
    ("schema", "Convert daily order revenue from EUR to USD"),
]

for mode, q in smoke_questions:
    print("\nQUERY:", q, "| mode:", mode)
    source_filter = {"database_schema"} if mode == "schema" else None
    res = retrieve_with_embeddings(
        export_model,
        updated_corpus,
        export_emb,
        q,
        top_k=10 if mode == "schema" else 5,
        device=DEVICE,
        source_filter=source_filter,
    )
    for r in res:
        print(f"  {r['rank']}. {r['source']} | {r['title']} | score={r['score']:.4f}")



QUERY: What is Apache Spark? | mode: global
  1. spark_docs | https://spark.apache.org/docs/latest/ | score=0.6267
  2. spark_docs | index.html | score=0.6089
  3. spark_docs | sparkr.html | score=0.6065
  4. spark_docs | JavaPairDStream.html | score=0.5907
  5. spark_docs | spark-connect-overview.html | score=0.5844

QUERY: How do I read a JSON file in Python? | mode: global
  1. codesearchnet | codesn_1532 | score=0.4633
  2. spark_docs | sparkr.html | score=0.4580
  3. codesearchnet | codesn_2992 | score=0.3962
  4. codesearchnet | codesn_2054 | score=0.3856
  5. codesearchnet | codesn_42 | score=0.3816

QUERY: Show revenue by customer segment | mode: schema
  1. database_schema | rag_kg.customers | score=0.5499
  2. database_schema | rag_kg.orders | score=0.4686
  3. database_schema | rag_kg.web_events | score=0.3630
  4. database_schema | rag_kg.campaigns | score=0.3558
  5. database_schema | rag_kg.support_tickets | score=0.3269
  6. database_schema | rag_kg.stores | score=0.325